--- Day 16: Reindeer Maze ---
It's time again for the Reindeer Olympics! This year, the big event is the Reindeer Maze, where the Reindeer compete for the lowest score.

You and The Historians arrive to search for the Chief right as the event is about to start. It wouldn't hurt to watch a little, right?

The Reindeer start on the Start Tile (marked S) facing East and need to reach the End Tile (marked E). They can move forward one tile at a time (increasing their score by 1 point), but never into a wall (#). They can also rotate clockwise or counterclockwise 90 degrees at a time (increasing their score by 1000 points).

To figure out the best place to sit, you start by grabbing a map (your puzzle input) from a nearby kiosk. For example:

###############
#.......#....E#
#.#.###.#.###.#
#.....#.#...#.#
#.###.#####.#.#
#.#.#.......#.#
#.#.#####.###.#
#...........#.#
###.#.#####.#.#
#...#.....#.#.#
#.#.#.###.#.#.#
#.....#...#.#.#
#.###.#.#.#.#.#
#S..#.....#...#
###############
There are many paths through this maze, but taking any of the best paths would incur a score of only 7036. This can be achieved by taking a total of 36 steps forward and turning 90 degrees a total of 7 times:


###############
#.......#....E#
#.#.###.#.###^#
#.....#.#...#^#
#.###.#####.#^#
#.#.#.......#^#
#.#.#####.###^#
#..>>>>>>>>v#^#
###^#.#####v#^#
#>>^#.....#v#^#
#^#.#.###.#v#^#
#^....#...#v#^#
#^###.#.#.#v#^#
#S..#.....#>>^#
###############
Here's a second example:

#################
#...#...#...#..E#
#.#.#.#.#.#.#.#.#
#.#.#.#...#...#.#
#.#.#.#.###.#.#.#
#...#.#.#.....#.#
#.#.#.#.#.#####.#
#.#...#.#.#.....#
#.#.#####.#.###.#
#.#.#.......#...#
#.#.###.#####.###
#.#.#...#.....#.#
#.#.#.#####.###.#
#.#.#.........#.#
#.#.#.#########.#
#S#.............#
#################
In this maze, the best paths cost 11048 points; following one such path would look like this:

#################
#...#...#...#..E#
#.#.#.#.#.#.#.#^#
#.#.#.#...#...#^#
#.#.#.#.###.#.#^#
#>>v#.#.#.....#^#
#^#v#.#.#.#####^#
#^#v..#.#.#>>>>^#
#^#v#####.#^###.#
#^#v#..>>>>^#...#
#^#v###^#####.###
#^#v#>>^#.....#.#
#^#v#^#####.###.#
#^#v#^........#.#
#^#v#^#########.#
#S#>>^..........#
#################
Note that the path shown above includes one 90 degree turn as the very first move, rotating the Reindeer from facing East to facing North.

Analyze your map carefully. What is the lowest score a Reindeer could possibly get?

In [93]:
from collections import deque

In [94]:
grid = {}
with open('input16.txt') as file:
    for i, line in enumerate(file):
        for j, char in enumerate(line):
            if char == "#":
                grid[(i, j)] = "#"
            elif char == ".":
                grid[(i, j)] = {}
            elif char == "S":
                grid[(i, j)] = {(i, j+1) : 0}
                start_tile = (i,j)
            elif char == "E":
                grid[(i, j)] = {}
                end_tile = (i,j)
grid, start_tile, end_tile

({(0, 0): '#',
  (0, 1): '#',
  (0, 2): '#',
  (0, 3): '#',
  (0, 4): '#',
  (0, 5): '#',
  (0, 6): '#',
  (0, 7): '#',
  (0, 8): '#',
  (0, 9): '#',
  (0, 10): '#',
  (0, 11): '#',
  (0, 12): '#',
  (0, 13): '#',
  (0, 14): '#',
  (0, 15): '#',
  (0, 16): '#',
  (0, 17): '#',
  (0, 18): '#',
  (0, 19): '#',
  (0, 20): '#',
  (0, 21): '#',
  (0, 22): '#',
  (0, 23): '#',
  (0, 24): '#',
  (0, 25): '#',
  (0, 26): '#',
  (0, 27): '#',
  (0, 28): '#',
  (0, 29): '#',
  (0, 30): '#',
  (0, 31): '#',
  (0, 32): '#',
  (0, 33): '#',
  (0, 34): '#',
  (0, 35): '#',
  (0, 36): '#',
  (0, 37): '#',
  (0, 38): '#',
  (0, 39): '#',
  (0, 40): '#',
  (0, 41): '#',
  (0, 42): '#',
  (0, 43): '#',
  (0, 44): '#',
  (0, 45): '#',
  (0, 46): '#',
  (0, 47): '#',
  (0, 48): '#',
  (0, 49): '#',
  (0, 50): '#',
  (0, 51): '#',
  (0, 52): '#',
  (0, 53): '#',
  (0, 54): '#',
  (0, 55): '#',
  (0, 56): '#',
  (0, 57): '#',
  (0, 58): '#',
  (0, 59): '#',
  (0, 60): '#',
  (0, 61): '#',
  (0, 62): '#',
  

In [95]:
queue = deque()
queue.append(start_tile)
while queue:
    i, j = start = queue.popleft()
    candidates = [(i, j+1), (i, j-1), (i+1, j), (i-1, j)] # E, W, S, N
    for c in candidates:
        if grid[c] == "#":
            continue

        # calculate scores:
        new_score = None
        for dir in grid[start]:
            if dir == c:
                score = grid[start][dir] + 1
            elif abs(dir[0] - c[0]) == 1: # 90° turn
                score = grid[start][dir] + 1001
            else: # 180° turn
                score = grid[start][dir] + 2001
            if new_score is None or score < new_score:
                new_score = score
        new_dir = tuple((c_coord + (c_coord - s_coord)) for s_coord, c_coord in zip (start, c))
        # Check for better scores:
        if new_dir in grid[c] and grid[c][new_dir] <= new_score:
            continue
        else:
            # Assign scores and populate queue:
            grid[c][new_dir] = new_score
            queue.append(c)
grid[end_tile]

{(0, 139): 107512}

Your puzzle answer was 107512.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
Now that you know what the best paths look like, you can figure out the best spot to sit.

Every non-wall tile (S, ., or E) is equipped with places to sit along the edges of the tile. While determining which of these tiles would be the best spot to sit depends on a whole bunch of factors (how comfortable the seats are, how far away the bathrooms are, whether there's a pillar blocking your view, etc.), the most important factor is whether the tile is on one of the best paths through the maze. If you sit somewhere else, you'd miss all the action!

So, you'll need to determine which tiles are part of any best path through the maze, including the S and E tiles.

In the first example, there are 45 tiles (marked O) that are part of at least one of the various best paths through the maze:

###############
#.......#....O#
#.#.###.#.###O#
#.....#.#...#O#
#.###.#####.#O#
#.#.#.......#O#
#.#.#####.###O#
#..OOOOOOOOO#O#
###O#O#####O#O#
#OOO#O....#O#O#
#O#O#O###.#O#O#
#OOOOO#...#O#O#
#O###.#.#.#O#O#
#O..#.....#OOO#
###############
In the second example, there are 64 tiles that are part of at least one of the best paths:

#################
#...#...#...#..O#
#.#.#.#.#.#.#.#O#
#.#.#.#...#...#O#
#.#.#.#.###.#.#O#
#OOO#.#.#.....#O#
#O#O#.#.#.#####O#
#O#O..#.#.#OOOOO#
#O#O#####.#O###O#
#O#O#..OOOOO#OOO#
#O#O###O#####O###
#O#O#OOO#..OOO#.#
#O#O#O#####O###.#
#O#O#OOOOOOO..#.#
#O#O#O#########.#
#O#OOO..........#
#################
Analyze your map further. How many tiles are part of at least one of the best paths through the maze?

In [96]:
"""
Ok. Working with what we have:  Starting at end_tile, we trace back the headings: e.g. if heading on end_tile is NORTH, we go SOUTH. Every tile we visit like this gets a flag on_path : True
if a tile has multiple directions stored, we calculate which path is cheaper and only pursue that one. If there is a tie, we pursue both paths.
"""

'\nOk. Working with what we have:  Starting at end_tile, we trace back the headings: e.g. if heading on end_tile is NORTH, we go SOUTH. Every tile we visit like this gets a flag on_path : True\nif a tile has multiple directions stored, we calculate which path is cheaper and only pursue that one. If there is a tie, we pursue both paths.\n'

In [97]:
def extend(orig, target):
    return tuple(2 * t - o for o, t in zip(orig, target))

def traceback(tile, facing=None):
    #scoring:
    scores = {}
    for heading in grid[tile]:
        if isinstance(heading, tuple) and len(heading) == 2:
            if not facing:
                score = grid[tile][heading]
            else:
                if heading == facing:
                    score = grid[tile][heading] + 1
                elif abs(heading[0] - facing[0]) == 1:
                    score = grid[tile][heading] + 1001
                else:
                    score = grid[tile][heading] + 2001
            scores[heading] = score
    
    # collect paths:
    new_tiles = [extend(heading, tile) for heading, score in scores.items() if score == min(scores.values())]
    return new_tiles

In [98]:
queue = deque()
path_tiles = set()
path_tiles.add(end_tile)
queue.append((end_tile, None))
while queue:
    tile, facing = queue.popleft()
    new_tiles = traceback(tile, facing)
    for n in new_tiles:
        if grid[n] != "#":
            path_tiles.add(n)
            queue.append((n, tile))
len(path_tiles)

561

In [99]:
for i in range(141):
    for j in range(141):
        if grid[(i, j)] == "#":
            print("#", end="")
        else:
            if (i, j) in path_tiles:
                print("O", end="")
            else:
                print(".", end="")
    print("")

#############################################################################################################################################
#.........#OOOOOOOOO#..OOOOO....#.....#OOOOOOOOOOOOO#OOOOOOOOOOOOOOOOOOOOOOOOO#.........#.............#...#.#.........#OOOOOOOOOOOOOOOOOOO#O#
#.#.#####.#O###.###O###O###O#.#.#.#.###O#########.#O#O###.#######.#.#########O#.#####.#.#.#####.#####.#.#.#.#.###.#.#.#O#######.#########O#O#
#.#.....#..O....#.#OOOOO#..O....#.#.#..O#...#.#...#O#O..#.#.#...#.#.#.....#.#O#.#.....#.#.#...#.....#...#.#.....#...#.#O#.#...#.....#...#O#O#
#.#####.###O#####.#.#######O#.###.#.#.#O#.#.#.#.###O#O#.#.#.#.#.#.#.#.###.#.#O###.###.#.#.###.#####.#####.###.###.#####O#.#.#.#####.#.#.#O#O#
#.....#....O#...#...#......O#.#...#.#..O#.#...#...#OOO#.#.#...#...#...#...#..O....#.#...#.#.....#...#...#...#.#...#OOOOO..#.#.....#.#.#OOO#O#
#########.#O#.###.###.#####O###.###.###O#.#.#####.###.#.#.#.###########.#####O#####.#.#.#.#.###.#.###.#.###.###.#.#O#######.#####.#.###O###O#
#.....

That's the right answer! You are one gold star closer to finding the Chief Historian.